# Day 21 — Task: Data Pipeline Integration

This notebook verifies our refactored data modules (`src.cleaning` and `src.features`) by processing a messy simulation dataset. We verify that all operations execute correctly and handle edge cases safely.

In [1]:
import pandas as pd
import numpy as np

# Import refactored modules
from src.cleaning import remove_duplicates, standardize_categories, coerce_numeric, quality_report
from src.features import extract_datetime_features, calculate_interaction_ratio, standard_scale

# Create a messy simulation dataset
raw_data = pd.DataFrame({
    'customer_id': [101, 102, 102, 103, 104, 105],
    'signup_date': ['2026-06-01 08:30:00', '2026-06-02 14:15:00', '2026-06-02 14:15:00', '2026-06-03 21:00:00', '2026-06-05 11:30:00', '2026-06-07 19:45:00'],
    'membership_tier': [' Silver ', 'Gold', 'Gold', 'Bronze', ' silver', 'GOLD'],
    'spending': ['120.5', '350.0', '350.0', 'invalid_cost', '80.0', '1500.0'], # has invalid numeric and an outlier (1500)
    'clicks': [10, 25, 25, 0, 8, 40] # has zero to test safe division
})

print("=== Messy Simulation Dataset ===")
print(raw_data)

=== Messy Simulation Dataset ===
   customer_id          signup_date membership_tier      spending  clicks
0          101  2026-06-01 08:30:00         Silver          120.5      10
1          102  2026-06-02 14:15:00            Gold         350.0      25
2          102  2026-06-02 14:15:00            Gold         350.0      25
3          103  2026-06-03 21:00:00          Bronze  invalid_cost       0
4          104  2026-06-05 11:30:00          silver          80.0       8
5          105  2026-06-07 19:45:00            GOLD        1500.0      40


In [2]:
print("=== Quality Report (Before) ===")
print(quality_report(raw_data))

# 1. Remove duplicate rows
cleaned_df = remove_duplicates(raw_data, subset=['customer_id', 'signup_date'])

# 2. Standardize categorical text values
cleaned_df['membership_tier'] = standardize_categories(cleaned_df['membership_tier'])

# 3. Parse and coerce numeric data
cleaned_df['spending'] = coerce_numeric(cleaned_df['spending'])

print("\n=== Quality Report (After Cleaning) ===")
print(quality_report(cleaned_df, numeric_columns=['spending']))
print("\n=== Cleaned Dataset ===")
print(cleaned_df)

=== Quality Report (Before) ===
           Metric  Value
0            Rows      6
1         Columns      5
2  Missing Values      0
3  Duplicate Rows      1

=== Quality Report (After Cleaning) ===
           Metric  Value
0            Rows      5
1         Columns      5
2  Missing Values      1
3  Duplicate Rows      0
4   Outlier Count      1

=== Cleaned Dataset ===
   customer_id          signup_date membership_tier  spending  clicks
0          101  2026-06-01 08:30:00          silver     120.5      10
1          102  2026-06-02 14:15:00            gold     350.0      25
3          103  2026-06-03 21:00:00          bronze       NaN       0
4          104  2026-06-05 11:30:00          silver      80.0       8
5          105  2026-06-07 19:45:00            gold    1500.0      40


C:\Users\Kavya\AppData\Local\Temp\ipykernel_6136\396328707.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['membership_tier'] = standardize_categories(cleaned_df['membership_tier'])
C:\Users\Kavya\AppData\Local\Temp\ipykernel_6136\396328707.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['spending'] = coerce_numeric(cleaned_df['spending'])


In [3]:
# 1. Extract rich datetime features
featured_df = extract_datetime_features(cleaned_df, 'signup_date')

# 2. Safely calculate interaction ratios (spending per click)
featured_df = calculate_interaction_ratio(featured_df, 'spending', 'clicks', 'spending_per_click')

# 3. Standard scale numerical values (spending)
featured_df = standard_scale(featured_df, ['spending'])

print("=== Final Transformed Dataset ===")
print(featured_df)

=== Final Transformed Dataset ===
   customer_id          signup_date membership_tier  spending  clicks  \
0          101  2026-06-01 08:30:00          silver -0.676912      10   
1          102  2026-06-02 14:15:00            gold -0.280734      25   
3          103  2026-06-03 21:00:00          bronze       NaN       0   
4          104  2026-06-05 11:30:00          silver -0.746826       8   
5          105  2026-06-07 19:45:00            gold  1.704473      40   

   signup_date_hour  signup_date_day_of_week signup_date_day_name  \
0                 8                        0               Monday   
1                14                        1              Tuesday   
3                21                        2            Wednesday   
4                11                        4               Friday   
5                19                        6               Sunday   

   signup_date_is_weekend  spending_per_click  
0                       0               12.05  
1               